# 示例: 运行带水泵的河道水力模型

**脚本:** `examples/run_pump_model.py`

## 目的
此示例用于演示如何在`preissmann_model`中设置并运行一个包含水泵（Pump）的水力模型。水泵作为一种内部边界条件，可以在河道中的某个点强行提升水位。

此笔记本将展示如何：
1.  定义一个水泵结构，包括其位置和性能曲线。
2.  将水泵添加到一个`HydraulicModel`实例中。
3.  运行一个模拟，其中水泵将水从低水位处提升到高水位处。
4.  检查并可视化最终的水面线和流量，以验证水泵的效果。

## 1. 环境设置

首先，我们导入所需的模型组件。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection
from preissmann_model.structures import Pump

## 2. 定义河道和水泵

我们创建一个平坦的河道，并在其中间节点（节点5）处定义一个水泵。水泵的性能由一条“扬程-流量”关系曲线`H = aQ^2 + bQ + c`来定义。这里我们定义的曲线是 `H = 5 - 0.05*Q^2`，意味着在流量为0时，水泵能提供5米的扬程；随着流量增大，扬程会减小。

In [ ]:
num_nodes = 11
reach_geom = RiverReach(
    cross_sections=[RectangularCrossSection(width=5) for _ in range(num_nodes)],
    lengths=np.full(num_nodes - 1, 100.0),
    slope=0.0, # 平坦河道
    manning_n=0.02
)

# 定义水泵
pump = Pump(name="main_pump", node_index=5, curve_coeffs=(-0.05, 0, 5.0))
print(f"定义水泵: '{pump.name}' @ 节点 {pump.node_index}")

## 3. 创建水力模型

我们创建一个`HydraulicModel`实例，并将上面创建的水泵作为`structures`参数传递进去。我们设置一个特殊的边界条件：下游水位（6.0米）高于上游水位（2.0米）。在没有水泵的情况下，水是无法从上游流到下游的。

In [ ]:
upstream_level = 2.0
downstream_level = 6.0

river = HydraulicModel(
    name="R1_pumped_reach",
    reach=reach_geom,
    dt=10.0,
    downstream_level=downstream_level,
    structures=[pump]
)

# 设置初始条件: 静水
river.Q[:] = 0.0
river.Z[:] = upstream_level
print(f"创建水力模型: '{river.name}'")

## 4. 运行模拟

我们运行一个足够长的模拟，让水流达到稳定状态。在这个例子中，我们使用水位作为上游边界条件（`Z_inflow`）。

In [ ]:
num_steps = 100
inflow_hydrograph = np.full(num_steps, upstream_level)
downstream_hydrograph = np.full(num_steps, downstream_level)

print(f"运行模拟 {num_steps} 步...")
for i in range(num_steps):
    inflows = {'Z_inflow': inflow_hydrograph[i]}
    river.downstream_level = downstream_hydrograph[i]
    river.step(inflows, river.dt)
print("模拟完成。")

## 5. 检查并可视化结果

模拟结束后，我们打印出每个节点的最终水位和流量，并绘制水面线和流量过程图。

从结果中可以看到：
- 在水泵所在位置（节点5），水位有一个明显的跃升，这正是水泵扬程的体现。
- 尽管下游水位高于上游，但在水泵的作用下，仍然在整个河道中产生了稳定的流量。

In [ ]:
print("\n--- 最终模型状态 ---")
print("节点 | 水位 (m) | 流量 (m^3/s)")
print("---- | --------- | ------------")
for i in range(river.num_nodes):
    print(f"{i:4d} | {river.Z[i]:9.3f} | {river.Q[i]:12.3f}")

# 可视化
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

node_indices = np.arange(river.num_nodes)

# 绘制水面线
ax1.plot(node_indices, river.Z, 'b-o', label='Water Surface Elevation')
ax1.set_ylabel('Elevation (m)')
ax1.set_title('Final Water Profile')
ax1.grid(True)

# 绘制流量
ax2.plot(node_indices, river.Q, 'r-s', label='Discharge')
ax2.set_xlabel('Node Index')
ax2.set_ylabel('Discharge (m³/s)')
ax2.set_title('Final Discharge')
ax2.grid(True)

plt.tight_layout()
plt.show()